# 02: Demonstrasi Google Colab — Simulasi Batch Peramalan Permintaan & BigQuery Sandbox
## Sistem Pendukung Keputusan Pengadaan & Perencanaan Produksi Bakery ITA Cake Shop Antapani

**Penulis / Mahasiswa:** Fadli (Skripsi Sistem Informasi / Teknik Informatika)  
**Tujuan Notebook:**  
Notebook interaktif ini disediakan khusus untuk demonstrasi langsung di hadapan Dosen Pembimbing dan Dewan Penguji Sidang Skripsi. Notebook ini menyimulasikan seluruh tahapan pipeline data otomatis:
1. **Otentikasi Cloud Bebas Kunci:** Menggunakan akun Google pribadi via `google.colab.auth.authenticate_user()` tanpa perlu mengekspos berkas kredensial sensitif.
2. **Pembersihan Data Kasir POS Olsera:** Koreksi finansial anti-double-discount, eliminasi order void, dan deduplikasi komposit.
3. **Pembangunan Kimball Data Mart (Star Schema):** Pembentukan tabel dimensi `dim_date`, `dim_product`, dan fakta `fact_sales`.
4. **Peramalan Ekonometrika Holt-Winters Multiplicative Terkunci:** Eksekusi parameter Champion empiris hasil Grid Search ($m=26$ untuk Cake & Japanese, $m=8$ untuk Mille Crepes, $\gamma=0.0000$).
5. **Disagregasi Anti-Limbah Makanan:** Porsi hari adaptif 4 pekan (28 hari), porsi SKU dinamis 12 pekan (84 hari), Service Level 95% ($Z=1.65$), dan MTO Threshold ($<0.5 \to 0$, `np.round`, tanpa `np.ceil`).
6. **Sinkronisasi BigQuery Sandbox:** Pemuatan idempoten via `WRITE_TRUNCATE` (mereset masa kedaluwarsa 60 hari) dan pembaruan SQL Views untuk Looker Studio secara *real-time*.

---
## 1. Instalasi Pustaka & Konfigurasi Lingkungan (Dependencies Setup)
Jalankan sel di bawah ini jika menjalankan notebook di runtime Google Colab untuk memastikan seluruh pustaka sains data dan Google Cloud BigQuery terpasang secara mutakhir.

In [ ]:
# Instalasi dependensi jika berjalan di Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("[*] Berjalan di Google Colab runtime. Memeriksa paket pendukung...")
    !pip install --quiet --upgrade google-cloud-bigquery statsmodels openpyxl pyyaml
    print("[*] Dependensi Google Colab berhasil disiapkan.")
except ImportError:
    IN_COLAB = False
    print("[*] Berjalan di lingkungan lokal / server mandiri.")

import os
import sys
import datetime
import yaml
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

print(f"[*] Python Version: {sys.version.split()[0]}")
print(f"[*] Pandas Version: {pd.__version__}")
print(f"[*] NumPy Version: {np.__version__}")

---
## 2. Otentikasi Pengguna Google Cloud Bebas Kunci (Keyless Cloud Auth)
Pada sesi sidang atau pengujian mandiri, Anda tidak perlu mengunggah file kunci JSON Service Account yang berisiko bocor. Cukup jalankan fungsi otentikasi bawaan Colab `google.colab.auth.authenticate_user()` untuk login langsung menggunakan akun Google yang terdaftar pada proyek BigQuery Anda.

In [ ]:
# Parameter Proyek GCP Anda
GCP_PROJECT_ID = "ita-cake-shop-bi"  # Ganti dengan Project ID GCP Anda jika berbeda
DATASET_ID = "ita_cake_shop_datamart"
LOCATION = "asia-southeast2"         # Jakarta (atau 'US' jika default sandbox)

if IN_COLAB:
    from google.colab import auth
    print("[*] Memulai otentikasi interaktif Google Cloud...")
    auth.authenticate_user()
    print("[+] Otentikasi berhasil! Akun Google telah terhubung.")

from google.cloud import bigquery
client = bigquery.Client(project=GCP_PROJECT_ID)
print(f"[+] BigQuery Client aktif untuk Project: {client.project}")

---
## 3. Konfigurasi Parameter Model Champion Ekonometrika Terkunci
Sesuai hasil pembuktian ilmiah pada Bab 4 skripsi (`03_Teori_Sparsitas_dan_GridSearch.ipynb`), parameter model *Holt-Winters Multiplicative* dikunci (*locked*) di lingkungan produksi untuk mencegah *dynamic seasonal overfitting* dan kegagalan konvergensi:

| Kategori | Model | Periode Musiman ($m$) | Level ($\alpha$) | Musiman ($\gamma$) | Tren | Composite WMAPE |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Cake** | HW Multiplicative | $m=26$ (Semesteran) | $0.4489$ | $0.0000$ (Deterministik) | None | **16.50%** |
| **Japanese** | HW Multiplicative | $m=26$ (Semesteran) | $0.3995$ | $0.0000$ (Deterministik) | None | **19.87%** |
| **Mille Crepes** | HW Multiplicative | $m=8$ (~2 Bulan) | $0.4977$ | $0.0000$ (Deterministik) | None | **18.00%** |

In [ ]:
CHAMPION_MODELS = {
    'Cake': {
        'model': 'Holt-Winters Mul', 'seasonal': 'mul', 'm': 26, 'trend': None,
        'alpha': 0.4489, 'beta': 0.0, 'gamma': 0.0, 'init': 'estimated',
        'wmape_holdout': 19.26, 'wmape_cv': 13.75, 'composite_wmape': 16.50
    },
    'Japanese': {
        'model': 'Holt-Winters Mul', 'seasonal': 'mul', 'm': 26, 'trend': None,
        'alpha': 0.3995, 'beta': 0.0, 'gamma': 0.0, 'init': 'estimated',
        'wmape_holdout': 15.33, 'wmape_cv': 24.42, 'composite_wmape': 19.87
    },
    'Mille Crepes': {
        'model': 'Holt-Winters Mul', 'seasonal': 'mul', 'm': 8, 'trend': None,
        'alpha': 0.4977, 'beta': 0.0, 'gamma': 0.0, 'init': 'estimated',
        'wmape_holdout': 21.68, 'wmape_cv': 14.32, 'composite_wmape': 18.00
    }
}

IDUL_FITRI_RANGES = [
    (pd.Timestamp('2023-04-08'), pd.Timestamp('2023-04-25')),
    (pd.Timestamp('2024-03-27'), pd.Timestamp('2024-04-13')),
    (pd.Timestamp('2025-03-17'), pd.Timestamp('2025-04-03')),
    (pd.Timestamp('2026-03-07'), pd.Timestamp('2026-03-24')),
]

IQR_CAPPING = {
    'Cake': 215.25,
    'Japanese': 216.38,
    'Mille Crepes': 180.25
}

TARGET_CATEGORIES = ['Cake', 'Japanese', 'Mille Crepes']
OFFICIAL_START = pd.Timestamp('2023-02-20')

print("[+] Parameter Champion Holt-Winters & Aturan Pra-Pemrosesan siap digunakan.")

---
## 4. Pemuatan & Pembersihan Data Transaksi POS (Olsera)
Data transaksi kasir dibersihkan melalui fungsi `clean_pos_dataset()`:
1. **Koreksi Finansial Anti-Double-Discount:** Pada Olsera POS, kolom `amount` sudah merupakan nilai setelah diskon. Rumus yang benar adalah `omzet_bersih = amount`.
2. **Eliminasi Transaksi Void:** Hanya menyaring order dengan status `'Z'` (tuntas).
3. **Deduplikasi Komposit:** Menghapus duplikasi teknis baris kasir dengan mempertahankan penjualan sah multi-baris.

In [ ]:
# Cari file Master Clean atau POS lokal
import glob

candidate_paths = [
    "Data_Output/Master_Clean_Feb2023_Sept2026.csv",
    "../01_data_ITA-Cake-Shop/Data_Output/Master_Clean_Feb2023_Sept2026.csv",
    "../Data_Output/Master_Clean_Feb2023_Sept2026.csv",
    "Master_Clean_Feb2023_Sept2026.csv"
]

data_file = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_file:
    print(f"[*] Membaca data dari master file: {data_file}")
    df_clean = pd.read_csv(data_file)
    df_clean['order date'] = pd.to_datetime(df_clean['order date'])
    if 'order no' in df_clean.columns:
        df_clean.rename(columns={'order no': 'order_no'}, inplace=True)
else:
    print("[!] Master file lokal tidak ditemukan. Mensimulasikan pemuatan dari file POS mentah...")
    # Anda dapat mengunggah file Excel mentah secara langsung via Colab file uploader jika diperlukan
    raise FileNotFoundError("Silakan tempatkan file Master_Clean_Feb2023_Sept2026.csv di direktori kerja.")

print(f"[+] Total baris transaksi bersih siap diproses: {len(df_clean):,} baris.")
print(f"[+] Rentang Waktu Transaksi: {df_clean['order date'].min().date()} s.d. {df_clean['order date'].max().date()}")

---
## 5. Pembangunan Kimball Data Mart (Star Schema)
Data transaksi dipetakan ke dalam arsitektur Kimball Data Mart:
1. **`dim_date`:** Tabel dimensi kalender komprehensif harian, dilengkapi penanda hari libur dan jendela Idul Fitri.
2. **`dim_product`:** Tabel dimensi produk (SCD Type 1) dengan flag `is_active_12w` untuk memisahkan SKU aktif vs SKU mati (*zero sales 84 hari*).
3. **`fact_sales`:** Tabel fakta penjualan granular tingkat transaksi.

In [ ]:
cutoff_dt = df_clean['order date'].max().normalize()
if cutoff_dt.dayofweek == 6:
    forecast_week_start = cutoff_dt + pd.Timedelta(days=1)
else:
    forecast_week_start = cutoff_dt + pd.Timedelta(days=(7 - cutoff_dt.dayofweek))

horizon_end = forecast_week_start + pd.Timedelta(days=6)

# 1. Pembangunan DIM_DATE
cal_dates = pd.date_range(OFFICIAL_START, horizon_end, freq='D')
dim_date = pd.DataFrame({'tanggal_penuh': cal_dates})
dim_date['date_id'] = dim_date['tanggal_penuh'].dt.strftime('%Y%m%d').astype(int)
dim_date['nama_hari'] = dim_date['tanggal_penuh'].dt.day_name()
dim_date['nama_bulan'] = dim_date['tanggal_penuh'].dt.month_name()
dim_date['minggu_ke'] = dim_date['tanggal_penuh'].dt.isocalendar().week.astype(int)
dim_date['tahun'] = dim_date['tanggal_penuh'].dt.year.astype(int)
dim_date['is_weekend'] = dim_date['tanggal_penuh'].dt.dayofweek.isin([5, 6])
dim_date['is_idulfitri_mask_window'] = False

for s_dt, e_dt in IDUL_FITRI_RANGES:
    mask = (dim_date['tanggal_penuh'] >= s_dt) & (dim_date['tanggal_penuh'] <= e_dt)
    dim_date.loc[mask, 'is_idulfitri_mask_window'] = True

dim_date['tanggal_penuh'] = dim_date['tanggal_penuh'].dt.strftime('%Y-%m-%d')
dim_date = dim_date[['date_id', 'tanggal_penuh', 'nama_hari', 'nama_bulan', 'minggu_ke', 'tahun', 'is_weekend', 'is_idulfitri_mask_window']]

# 2. Pembangunan DIM_PRODUCT
latest_prod = df_clean.sort_values('order date').groupby('item name').tail(1)
dim_product = latest_prod[['item name', 'item group', 'price']].copy()
dim_product.rename(columns={'item name': 'product_name', 'item group': 'category_name', 'price': 'sell_price_current'}, inplace=True)
dim_product.reset_index(drop=True, inplace=True)
dim_product['product_id'] = dim_product.index + 1
dim_product['scd_type'] = 'Type1'

cutoff_12w = horizon_end - pd.Timedelta(days=84)
target_clean = df_clean[df_clean['item group'].isin(TARGET_CATEGORIES)]
sales_12w = target_clean[target_clean['order date'] >= cutoff_12w].groupby('item name')['qty'].sum()
active_skus = set(sales_12w[sales_12w > 0].index)

dim_product['is_active_12w'] = dim_product['product_name'].apply(lambda x: True if x in active_skus else False).astype(object)
dim_product.loc[~dim_product['category_name'].isin(TARGET_CATEGORIES), 'is_active_12w'] = None

dim_product = dim_product[['product_id', 'product_name', 'category_name', 'sell_price_current', 'scd_type', 'is_active_12w']]

# 3. Pembangunan FACT_SALES
date_lookup = dict(zip(pd.to_datetime(dim_date['tanggal_penuh']).dt.date, dim_date['date_id']))
prod_lookup = dict(zip(dim_product['product_name'], dim_product['product_id']))

fact_sales = df_clean.copy()
fact_sales['date_id'] = fact_sales['order date'].dt.date.map(date_lookup)
fact_sales['product_id'] = fact_sales['item name'].map(prod_lookup)
fact_sales.dropna(subset=['date_id', 'product_id'], inplace=True)

fact_sales = fact_sales[['order_no', 'date_id', 'product_id', 'price', 'qty', 'gross_amount', 'discount amount', 'omzet_bersih']].copy()
fact_sales.columns = ['order_no', 'date_id', 'product_id', 'unit_price', 'qty_sold', 'gross_amount', 'discount_amount', 'net_amount']
fact_sales = fact_sales.astype({'date_id': int, 'product_id': int, 'qty_sold': int})
fact_sales['unit_price'] = fact_sales['unit_price'].round(2)
fact_sales['gross_amount'] = fact_sales['gross_amount'].round(2)
fact_sales['discount_amount'] = fact_sales['discount_amount'].round(2)
fact_sales['net_amount'] = fact_sales['net_amount'].round(2)

print(f"[+] DIM_DATE     : {len(dim_date):,} baris ({dim_date['tanggal_penuh'].min()} s.d. {dim_date['tanggal_penuh'].max()})")
print(f"[+] DIM_PRODUCT  : {len(dim_product):,} SKU terdaftar ({len(active_skus)} SKU Aktif)")
print(f"[+] FACT_SALES   : {len(fact_sales):,} transaksi terhubung (Total Net: Rp {fact_sales['net_amount'].sum():,.0f})")

---
## 6. Peramalan Holt-Winters Makro & Disagregasi Mikro Anti-Limbah
Model peramalan mingguan makro dijalankan dengan substitusi `replace(0, 0.1)` untuk keamanan optimasi log-likelihood multiplikatif, kemudian didisagregasikan ke target harian per SKU:
1. **Porsi Hari Adaptif:** Rata-rata 4 pekan terakhir (28 hari).
2. **Porsi SKU Dinamis:** Rata-rata *rolling window* 12 pekan terakhir (84 hari). Dilarang memakai rasio *all-time*.
3. **Persediaan Pengaman (Safety Stock):** $Z = 1.65 \times \frac{\sigma_{\text{resid}} \times w_{\text{SKU}}}{\sqrt{7}}$ (Service Level 95%).
4. **MTO Threshold (Pemutus Rantai Limbah Makanan):**
   - Jika $\text{Target}_{\text{raw}} < 0.5 \to \text{target} = 0$ (Made-To-Order / Pre-Order).
   - Jika $\text{Target}_{\text{raw}} \ge 0.5 \to \text{target} = \text{np.round}(\text{Target}_{\text{raw}})$.
   - Dilarang keras menggunakan `np.ceil()`.

In [ ]:
history_data = df_clean[
    (df_clean['order date'] < forecast_week_start) &
    (df_clean['item group'].isin(TARGET_CATEGORIES))
].copy()

# Resampling mingguan W-MON
date_idx = pd.date_range(history_data['order date'].min().normalize(), history_data['order date'].max().normalize(), freq='D')
daily_pivot = history_data.pivot_table(index='order date', columns='item group', values='qty', aggfunc='sum').reindex(date_idx, fill_value=0)
weekly_raw = daily_pivot.resample('W-MON', closed='left', label='left').sum()[TARGET_CATEGORIES]
weekly_clean = weekly_raw.copy()

# Masking Idul Fitri & Capping
for s_dt, e_dt in IDUL_FITRI_RANGES:
    mask = (weekly_clean.index + pd.Timedelta(days=6) >= s_dt) & (weekly_clean.index <= e_dt)
    weekly_clean.loc[mask, :] = np.nan
weekly_clean = weekly_clean.interpolate(method='linear').bfill().ffill()

for cat in TARGET_CATEGORIES:
    ub = IQR_CAPPING.get(cat, weekly_clean[cat].quantile(0.75) * 1.5)
    weekly_clean[cat] = np.where(weekly_clean[cat] > ub, ub, weekly_clean[cat])

# Model Fitting
weekly_forecasts = {}
residuals_std = {}
for cat in TARGET_CATEGORIES:
    series = weekly_clean[cat].copy().replace(0, 0.1)
    p = CHAMPION_MODELS[cat]
    model = ExponentialSmoothing(
        series, trend=p['trend'], seasonal=p['seasonal'],
        seasonal_periods=p['m'], initialization_method=p['init']
    )
    fitted = model.fit(
        smoothing_level=p['alpha'],
        smoothing_trend=None,
        smoothing_seasonal=p['gamma'],
        optimized=False
    )
    fc = float(max(0.0, fitted.forecast(1).values[0]))
    weekly_forecasts[cat] = fc
    residuals_std[cat] = float(np.std(fitted.resid))
    print(f"[*] Forecast Mingguan [{cat}]: {fc:.2f} pcs | Residual Std: {residuals_std[cat]:.2f}")

# Disagregasi Harian (7 Hari Kalender)
forecast_dates = pd.date_range(forecast_week_start, periods=7, freq='D')
daily_sku_target = df_clean[
    (df_clean['order date'] < forecast_week_start) &
    (df_clean['item group'].isin(TARGET_CATEGORIES))
].groupby(['order date', 'item group', 'item name'])['qty'].sum().reset_index()

plan_records = []
batch_ts = datetime.datetime.now(datetime.timezone.utc).isoformat()

for cat in TARGET_CATEGORIES:
    w_fc = weekly_forecasts[cat]
    std_resid = residuals_std[cat]
    
    # 4W DoW Share
    dow_start = forecast_week_start - pd.Timedelta(days=28)
    dow_data = daily_sku_target[(daily_sku_target['order date'] >= dow_start) & (daily_sku_target['order date'] < forecast_week_start) & (daily_sku_target['item group'] == cat)]
    tot_dow = dow_data['qty'].sum()
    dow_share = {i: dow_data.groupby(dow_data['order date'].dt.dayofweek)['qty'].sum().get(i, 0) / tot_dow for i in range(7)} if tot_dow > 0 else {i: 1/7 for i in range(7)}
    
    # 12W SKU Share
    sku_start = forecast_week_start - pd.Timedelta(days=84)
    sku_data = daily_sku_target[(daily_sku_target['order date'] >= sku_start) & (daily_sku_target['order date'] < forecast_week_start) & (daily_sku_target['item group'] == cat)]
    tot_sku = sku_data['qty'].sum()
    sku_share = (sku_data.groupby('item name')['qty'].sum() / tot_sku).to_dict() if tot_sku > 0 else {}
    
    for dt in forecast_dates:
        day_fc = w_fc * dow_share.get(dt.dayofweek, 1/7)
        for sku, share in sku_share.items():
            if share <= 0: continue
            sku_fc = day_fc * share
            ss = 1.65 * std_resid * share / np.sqrt(7)
            raw_tgt = sku_fc + ss
            final_tgt = 0 if raw_tgt < 0.5 else int(np.round(raw_tgt))
            
            plan_records.append({
                'forecast_date': dt.strftime('%Y-%m-%d'),
                'category_name': cat,
                'product_name': sku,
                'forecast_murni': int(np.round(sku_fc)),
                'safety_stock': int(np.round(ss)),
                'target_produksi': final_tgt,
                'batch_execution_timestamp': batch_ts
            })

plan_production_daily = pd.DataFrame(plan_records).sort_values(by=['forecast_date', 'target_produksi'], ascending=[True, False]).reset_index(drop=True)

# Rekomendasi Manajerial
forecast_recommendations = pd.DataFrame([
    {
        'kategori': cat, 'model_arsitektur': f"Holt-Winters {p['seasonal'].title()}",
        'periode_musiman_m': p['m'], 'smoothing_alpha': p['alpha'], 'smoothing_gamma': p['gamma'],
        'holdout_wmape': f"{p['wmape_holdout']:.2f}%", 'cv_wmape_3fold': f"{p['wmape_cv']:.2f}%",
        'composite_wmape': f"{p['composite_wmape']:.2f}%",
        'rekomendasi_operasional': 'Parameter stabil dan terbukti unggul terhadap benchmark Naive.'
    } for cat, p in CHAMPION_MODELS.items()
])

print(f"\n[+] Rencana Produksi Harian (plan_production_daily) berhasil dibentuk: {len(plan_production_daily):,} baris.")
print(f"[+] Total Target 7 Hari: {plan_production_daily['target_produksi'].sum():,} pcs "
      f"(Forecast: {plan_production_daily['forecast_murni'].sum():,} + SS: {plan_production_daily['safety_stock'].sum():,})")

---
## 7. Unggah ke Google BigQuery Sandbox & Pembuatan SQL Views
Mengunggah data layer ke BigQuery Sandbox menggunakan `client.load_table_from_dataframe()`:
- **Sandbox-Compatible:** Tanpa kartu kredit, tanpa DML berbayar, dan tanpa bucket Cloud Storage.
- **Idempotensi via `WRITE_TRUNCATE`:** Memuat ulang tabel gudang secara bersih dan otomatis memperpanjang masa kedaluwarsa tabel 60 hari setiap pekan.
- **SQL DDL Views:** Otomatis memperbarui view `v_looker_performance_overview` dan `v_looker_production_planning`.

In [ ]:
# Unggah DataFrame ke BigQuery Sandbox
tables_to_upload = {
    'dim_date': dim_date,
    'dim_product': dim_product,
    'fact_sales': fact_sales,
    'plan_production_daily': plan_production_daily,
    'forecast_recommendations': forecast_recommendations
}

dataset_ref = bigquery.DatasetReference(GCP_PROJECT_ID, DATASET_ID)

try:
    client.get_dataset(dataset_ref)
    print(f"[+] Dataset '{DATASET_ID}' ditemukan.")
except Exception:
    print(f"[*] Dataset '{DATASET_ID}' belum ada. Membuat dataset baru...")
    ds = bigquery.Dataset(dataset_ref)
    ds.location = LOCATION
    ds.default_table_expiration_ms = 60 * 24 * 3600 * 1000
    client.create_dataset(ds, exists_ok=True)
    print(f"[+] Dataset '{DATASET_ID}' berhasil dibuat di region {LOCATION}.")

job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

for table_name, df_tbl in tables_to_upload.items():
    t_ref = dataset_ref.table(table_name)
    print(f"[*] Mengunggah '{table_name}' ({len(df_tbl):,} baris) via WRITE_TRUNCATE...")
    job = client.load_table_from_dataframe(df_tbl, t_ref, job_config=job_config)
    job.result()
    print(f"[+] Tabel '{table_name}' berhasil dimuat!")

# Pembuatan SQL Views
views_ddl = {
    'v_looker_performance_overview': f"""
        CREATE OR REPLACE VIEW `{GCP_PROJECT_ID}.{DATASET_ID}.v_looker_performance_overview` AS
        SELECT
            f.order_no,
            PARSE_DATE('%Y-%m-%d', d.tanggal_penuh) AS order_date,
            d.nama_hari, d.nama_bulan, d.minggu_ke, d.tahun, d.is_weekend,
            p.product_id, p.product_name, p.category_name,
            f.qty_sold, f.gross_amount, f.discount_amount, f.net_amount
        FROM `{GCP_PROJECT_ID}.{DATASET_ID}.fact_sales` f
        JOIN `{GCP_PROJECT_ID}.{DATASET_ID}.dim_date` d ON f.date_id = d.date_id
        JOIN `{GCP_PROJECT_ID}.{DATASET_ID}.dim_product` p ON f.product_id = p.product_id;
    """,
    'v_looker_production_planning': f"""
        CREATE OR REPLACE VIEW `{GCP_PROJECT_ID}.{DATASET_ID}.v_looker_production_planning` AS
        SELECT
            PARSE_DATE('%Y-%m-%d', forecast_date) AS forecast_date,
            category_name, product_name, forecast_murni, safety_stock, target_produksi,
            FORMAT_DATE('%a, %d %b %Y', PARSE_DATE('%Y-%m-%d', forecast_date)) AS formatted_date,
            batch_execution_timestamp
        FROM `{GCP_PROJECT_ID}.{DATASET_ID}.plan_production_daily`
        ORDER BY forecast_date ASC, target_produksi DESC;
    """
}

for v_name, ddl in views_ddl.items():
    print(f"[*] Menjalankan DDL untuk View '{v_name}'...")
    q = client.query(ddl)
    q.result()
    print(f"[+] View '{v_name}' berhasil diperbarui!")

print("\n" + "=" * 60)
print("[+] SINKRONISASI BIGQUERY SANDBOX SELESAI DENGAN SUKSES!")
print("=" * 60)

---
## 8. Verifikasi Interaktif Query BigQuery & Looker Studio
Jalankan query verifikasi langsung dari Colab untuk membuktikan bahwa data mart telah aktif dan siap dihubungkan ke Looker Studio.

In [ ]:
# 1. Query KPI Halaman 1
sql_kpi = f"""
    SELECT
        ROUND(SUM(net_amount) / 1e9, 2) AS total_revenue_milyar,
        SUM(qty_sold) AS total_qty_sold,
        COUNT(DISTINCT order_no) AS total_orders
    FROM `{GCP_PROJECT_ID}.{DATASET_ID}.v_looker_performance_overview`
"""
df_kpi = client.query(sql_kpi).to_dataframe()
print("--- [VERIFIKASI KPI HALAMAN 1 LOOKER STUDIO] ---")
print(df_kpi.to_string(index=False))

# 2. Query Rencana Produksi Halaman 2 (Top 10 Hari Ini)
sql_plan = f"""
    SELECT
        formatted_date,
        category_name,
        product_name,
        forecast_murni,
        safety_stock,
        target_produksi
    FROM `{GCP_PROJECT_ID}.{DATASET_ID}.v_looker_production_planning`
    WHERE target_produksi > 0
    ORDER BY forecast_date ASC, target_produksi DESC
    LIMIT 10
"""
df_plan_preview = client.query(sql_plan).to_dataframe()
print("\n--- [VERIFIKASI TARGET PRODUKSI HALAMAN 2 (TOP 10 SKU AKTIF)] ---")
print(df_plan_preview.to_string(index=False))

---
## 9. Kesimpulan & Panduan Tindak Lanjut Sidang Skripsi

1. **Integritas Otomasi Terbukti:** Seluruh alur data dari file kasir hingga tabel data mart di BigQuery Sandbox telah berhasil dieksekusi secara otomatis, idempoten, dan bebas biaya.
2. **Kesesuaian dengan Dokumen Sidang:** Skema `v_looker_performance_overview` dan `v_looker_production_planning` terbukti memetakan 100% elemen visual pada berkas `Sidang/Prototype_dashboard_ITA_Cake_shop.pdf`.
3. **Langkah Berikutnya bagi Mahasiswa:**
   - Buka Google Looker Studio: 👉 **https://lookerstudio.google.com/**
   - Hubungkan sumber data ke Project `{GCP_PROJECT_ID}`, Dataset `{DATASET_ID}`.
   - Buka laporan yang telah dirakit untuk menampilkan visualisasi dinamis di hadapan dosen penguji.